In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time

# 1. Detect available devices
device_gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
print(f"GPU device: {device_gpu}")
print(f"CPU device: {device_cpu}")

# 2. Load and split the data
data = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# 3. Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# 4. Create PyTorch datasets & loaders
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=128)

# 5. Define a simple MLP
class MiniMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# 6. Function to train and evaluate model on a specific device
def train_and_evaluate(device_name, device):
    print(f"\n--- Running on {device_name} ---")
    
    # Initialize model on the device
    model = MiniMLP(input_dim=X_train.shape[1]).to(device)
    
    # Set up training
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    num_epochs = 5
    
    # Training loop
    train_start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        epoch_start_time = time.time()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        epoch_time = time.time() - epoch_start_time
        print(f"Epoch {epoch+1}/{num_epochs} — Train MSE: {avg_loss:.4f} — Time: {epoch_time:.2f}s")
    
    train_time = time.time() - train_start_time
    print(f"Total training time: {train_time:.2f}s")
    
    # Evaluation
    eval_start_time = time.time()
    model.eval()
    with torch.no_grad():
        total_loss = 0.0
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            total_loss += criterion(preds, yb).item() * xb.size(0)
        test_mse = total_loss / len(test_loader.dataset)
    eval_time = time.time() - eval_start_time
    print(f"Test MSE: {test_mse:.4f} — Evaluation time: {eval_time:.2f}s")
    
    return train_time, eval_time, test_mse

# 7. Run on GPU (if available)
if torch.cuda.is_available():
    gpu_train_time, gpu_eval_time, gpu_mse = train_and_evaluate("GPU", device_gpu)

# 8. Run on CPU
cpu_train_time, cpu_eval_time, cpu_mse = train_and_evaluate("CPU", device_cpu)

# 9. Compare results
if torch.cuda.is_available():
    print("\n--- Performance Comparison ---")
    print(f"GPU Training Time: {gpu_train_time:.2f}s | CPU Training Time: {cpu_train_time:.2f}s | Speedup: {cpu_train_time/gpu_train_time:.2f}x")
    print(f"GPU Evaluation Time: {gpu_eval_time:.2f}s | CPU Evaluation Time: {cpu_eval_time:.2f}s | Speedup: {cpu_eval_time/gpu_eval_time:.2f}x")
    print(f"GPU Test MSE: {gpu_mse:.4f} | CPU Test MSE: {cpu_mse:.4f}")


GPU device: cuda
CPU device: cpu

--- Running on GPU ---
Epoch 1/5 — Train MSE: 2.5185 — Time: 0.97s
Epoch 2/5 — Train MSE: 0.8660 — Time: 0.35s
Epoch 3/5 — Train MSE: 0.6782 — Time: 0.36s
Epoch 4/5 — Train MSE: 0.5574 — Time: 0.37s
Epoch 5/5 — Train MSE: 0.4925 — Time: 0.34s
Total training time: 2.40s
Test MSE: 0.4850 — Evaluation time: 0.05s

--- Running on CPU ---
Epoch 1/5 — Train MSE: 2.2824 — Time: 0.26s
Epoch 2/5 — Train MSE: 0.8084 — Time: 0.25s
Epoch 3/5 — Train MSE: 0.6409 — Time: 0.24s
Epoch 4/5 — Train MSE: 0.5485 — Time: 0.24s
Epoch 5/5 — Train MSE: 0.4941 — Time: 0.24s
Total training time: 1.24s
Test MSE: 0.4948 — Evaluation time: 0.04s

--- Performance Comparison ---
GPU Training Time: 2.40s | CPU Training Time: 1.24s | Speedup: 0.52x
GPU Evaluation Time: 0.05s | CPU Evaluation Time: 0.04s | Speedup: 0.86x
GPU Test MSE: 0.4850 | CPU Test MSE: 0.4948


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import optuna
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing           # ← changed
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import time
import numpy as np

seeds = [1,2,3]
iter = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Search space (updated)
param_grid = {
    "n_layers":      [1, 2, 3],
    "n_units_l0":    [32, 96, 160, 224],
    "n_units_l1":    [32, 96, 160, 224],
    "n_units_l2":    [32, 96, 160, 224],
    "activation":    ["relu", "tanh"],                           # ← added
    "optimizer":     ["adam", "sgd"],                            # ← added
    "learning_rate": [0.0001, 0.001, 0.01],                      # ← reduced values for grid search
    "batch_size":    [32, 96, 160, 224],
    "dropout_rate":  [0.0, 0.2, 0.4],                            # ← reduced values for grid search
    "weight_decay":  [1e-6, 1e-4, 1e-2],                         # ← reduced values for grid search
}

# 2) Data prep (regression)
X, y = fetch_california_housing(return_X_y=True, as_frame=False)  # ← changed
X = StandardScaler().fit_transform(X.astype('float32'))
y = y.astype('float32').reshape(-1,1)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=iter
)
train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))

# 3) Model (with dropout & activation selection)
class OptunaRegNet(nn.Module):
    def __init__(self, trial):
        super().__init__()
        # suggest depth & widths
        n_layers      = trial.suggest_categorical("n_layers", param_grid["n_layers"])
        hidden_sizes  = []
        if n_layers >= 1:
            hidden_sizes.append(trial.suggest_categorical("n_units_l0", param_grid["n_units_l0"]))
        if n_layers >= 2:
            hidden_sizes.append(trial.suggest_categorical("n_units_l1", param_grid["n_units_l1"]))
        if n_layers >= 3:
            hidden_sizes.append(trial.suggest_categorical("n_units_l2", param_grid["n_units_l2"]))

        self.activation   = nn.ReLU() if trial.suggest_categorical("activation", param_grid["activation"])=="relu" else nn.Tanh()  # ← added
        self.dropout_rate = trial.suggest_categorical("dropout_rate", param_grid["dropout_rate"])                                                    # ← added

        layers = []
        in_dim = X.shape[1]
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(self.activation)
            layers.append(nn.Dropout(self.dropout_rate))
            in_dim = h
        self.net    = nn.Sequential(*layers)
        self.output = nn.Linear(in_dim, 1)

    def forward(self, x):
        x = self.net(x)
        return self.output(x)

# 4) Objective (with float/log LR, optimizer choice & weight_decay)
def objective(trial):
    model = OptunaRegNet(trial).to(device)

    lr  = trial.suggest_categorical("learning_rate", param_grid["learning_rate"])                                               # ← changed
    wd  = trial.suggest_categorical("weight_decay", param_grid["weight_decay"])                                               # ← added
    bs  = trial.suggest_categorical("batch_size", param_grid["batch_size"])

    # choose optimizer
    opt_name = trial.suggest_categorical("optimizer", param_grid["optimizer"])
    optimizer = (torch.optim.Adam if opt_name=="adam" else torch.optim.SGD)(
        model.parameters(), lr=lr, weight_decay=wd                                                        # ← added wd
    )

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False)
    loss_fn      = nn.MSELoss(reduction='sum')                                                              # ← changed to regression

    # training
    try:
        for _ in range(30):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                outputs = model(xb)
                loss = loss_fn(outputs, yb)
                
                # Check for NaN loss
                if torch.isnan(loss).any():
                    print(f"NaN loss detected during training with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN loss encountered")
                
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                # Check for NaN weights after update
                for param in model.parameters():
                    if torch.isnan(param).any():
                        print(f"NaN weights detected after update with params: {trial.params}")
                        raise optuna.exceptions.TrialPruned("NaN weights encountered")

        # evaluation
        model.eval()
        total_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                
                # Check for NaN outputs
                if torch.isnan(outputs).any():
                    print(f"NaN outputs detected during evaluation with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN outputs encountered")
                
                batch_loss = loss_fn(outputs, yb).item()
                
                # Check for NaN loss
                if np.isnan(batch_loss):
                    print(f"NaN loss detected during evaluation with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN evaluation loss encountered")
                
                total_loss += batch_loss

        avg_mse = total_loss / len(test_ds)
        
        # Final check for NaN result
        if np.isnan(avg_mse):
            print(f"NaN final MSE with params: {trial.params}")
            raise optuna.exceptions.TrialPruned("NaN final MSE encountered")
            
        trial.set_user_attr("mse", avg_mse)
        return -avg_mse  # maximize negative MSE
    
    except Exception as e:
        print(f"Error in trial with params {trial.params}: {str(e)}")
        raise optuna.exceptions.TrialPruned(f"Trial failed with error: {str(e)}")

# 5) Logging callback remains the same, but record "mse"
results = []
def logging_callback(study, trial):
    entry = trial.params.copy()
    entry["neg_mse"]      = trial.value
    entry["trial_number"] = trial.number
    entry["trial_end"]    = time.time()
    entry["mse"]          = trial.user_attrs["mse"]
    results.append(entry)
    pbar.update(1)


In [2]:
import time
import json
from tqdm.auto import tqdm
from optuna.samplers import GridSampler
from sklearn.model_selection import ParameterGrid
import numpy as np

# Maximum runtime per grid search (1.75 hours in seconds)
MAX_RUNTIME = 1.75 * 60 * 60

# Seeds for reproducibility
grid_seeds = [0, 1, 2]

# Function to run a single grid search with time limit
def run_grid_search(seed_idx):
    global results
    
    # Set seed for reproducibility
    np.random.seed(grid_seeds[seed_idx])
    torch.manual_seed(grid_seeds[seed_idx])
    
    # Reset results for this run
    results = []
    
    # Create grid sampler
    sampler = GridSampler(param_grid)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    # Calculate number of trials in the grid
    n_trials = len(list(ParameterGrid(param_grid)))
    print(f"Running grid search {seed_idx+1}/3 with seed {grid_seeds[seed_idx]}")
    print(f"Grid contains {n_trials} configurations")
    
    # Setup progress bar based on time rather than iterations
    global pbar
    start_time = time.time()
    results.append({"start_time": start_time})
    pbar = tqdm(total=MAX_RUNTIME, desc=f"Grid Search (seed={grid_seeds[seed_idx]})")
    
    # Time-based progress bar update function
    def time_based_callback(study, trial):
        # Update the regular logging
        entry = trial.params.copy()
        entry["neg_mse"] = trial.value
        entry["trial_number"] = trial.number
        entry["trial_end"] = time.time()
        entry["mse"] = trial.user_attrs["mse"]
        results.append(entry)
        
        # Update progress bar based on elapsed time
        elapsed = time.time() - start_time
        pbar.n = min(elapsed, MAX_RUNTIME)
        pbar.refresh()
    
    # Run optimization with time limit
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=5,
        callbacks=[time_based_callback],
        timeout=MAX_RUNTIME
    )
    
    # Ensure progress bar is complete
    pbar.n = MAX_RUNTIME
    pbar.refresh()
    pbar.close()
    
    # Save results
    with open(f"results/GRID/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"Grid search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)
    
    return study.best_value, study.best_trial.params

# Create directory if it doesn't exist
import os
os.makedirs("results/GRID", exist_ok=True)

# Run all three grid searches sequentially
best_configs = []
for i in [1, 2]:
    best_value, best_params = run_grid_search(i)
    best_configs.append((best_value, best_params))


[I 2025-06-15 22:05:59,711] A new study created in memory with name: no-name-82baf8f7-a8a8-44c6-afa9-41df945501d2


Running grid search 2/3 with seed 1
Grid contains 82944 configurations


Grid Search (seed=1):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-15 22:06:33,336] Trial 4 finished with value: -3.383281944334045 and parameters: {'n_layers': 1, 'n_units_l0': 96, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001, 'weight_decay': 1e-06, 'batch_size': 224, 'optimizer': 'sgd'}. Best is trial 4 with value: -3.383281944334045.
[I 2025-06-15 22:06:47,688] Trial 3 finished with value: -0.2864577335904735 and parameters: {'n_layers': 2, 'n_units_l0': 160, 'n_units_l1': 160, 'activation': 'relu', 'dropout_rate': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 160, 'optimizer': 'adam'}. Best is trial 3 with value: -0.2864577335904735.
[I 2025-06-15 22:07:12,988] Trial 0 finished with value: -2.917484009912772 and parameters: {'n_layers': 3, 'n_units_l0': 96, 'n_units_l1': 224, 'n_units_l2': 32, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'batch_size': 96, 'optimizer': 'sgd'}. Best is trial 3 with value: -0.2864577335904735.
[I 2025-06-15 22:07:29,594

Grid search 2 completed
Best negative MSE: -0.2671715566931769
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 96, 'n_units_l1': 96, 'n_units_l2': 160, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'optimizer': 'adam'}
Runtime: 106.16 minutes
--------------------------------------------------
Running grid search 3/3 with seed 2
Grid contains 82944 configurations


Grid Search (seed=2):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-15 23:52:34,269] Trial 4 finished with value: -4.000103233396545 and parameters: {'n_layers': 1, 'n_units_l0': 96, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.0001, 'weight_decay': 1e-06, 'batch_size': 224, 'optimizer': 'sgd'}. Best is trial 4 with value: -4.000103233396545.
[I 2025-06-15 23:52:51,207] Trial 3 finished with value: -0.287727576817653 and parameters: {'n_layers': 2, 'n_units_l0': 160, 'n_units_l1': 160, 'activation': 'relu', 'dropout_rate': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 160, 'optimizer': 'adam'}. Best is trial 3 with value: -0.287727576817653.
[I 2025-06-15 23:53:06,978] Trial 0 finished with value: -2.4536920517914056 and parameters: {'n_layers': 3, 'n_units_l0': 96, 'n_units_l1': 224, 'n_units_l2': 32, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'batch_size': 96, 'optimizer': 'sgd'}. Best is trial 3 with value: -0.287727576817653.
[I 2025-06-15 23:53:27,850] 

Grid search 3 completed
Best negative MSE: -0.2660001636475556
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 96, 'n_units_l1': 96, 'n_units_l2': 160, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 32, 'optimizer': 'adam'}
Runtime: 105.69 minutes
--------------------------------------------------


In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import optuna
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import json
import time
import numpy as np

seeds = [1, 2, 3]
iter = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Search space with continuous parameters
param_grid = {
    "n_layers":      [1, 2, 3],
    "n_units_l0":    [32, 256],  # min and max bounds
    "n_units_l1":    [32, 256],  # min and max bounds
    "n_units_l2":    [32, 256],  # min and max bounds
    "activation":    ["relu", "tanh"],
    "optimizer":     ["adam", "sgd"],
    "learning_rate": [0.0001, 0.001, 0.01],
    "batch_size":    [32, 256],  # min and max bounds
    "dropout_rate":  [0.0, 0.2, 0.4],
    "weight_decay":  [1e-6, 1e-4, 1e-2],
}

# 2) Data prep (regression)
X, y = fetch_california_housing(return_X_y=True, as_frame=False)
X = StandardScaler().fit_transform(X.astype('float32'))
y = y.astype('float32').reshape(-1,1)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=iter
)
train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))

# 3) Model (with dropout & activation selection)
class OptunaRegNet(nn.Module):
    def __init__(self, trial):
        super().__init__()
        # suggest depth & widths
        n_layers      = trial.suggest_categorical("n_layers", param_grid["n_layers"])
        hidden_sizes  = []
        if n_layers >= 1:
            hidden_sizes.append(trial.suggest_int("n_units_l0", param_grid["n_units_l0"][0], param_grid["n_units_l0"][1]))
        if n_layers >= 2:
            hidden_sizes.append(trial.suggest_int("n_units_l1", param_grid["n_units_l1"][0], param_grid["n_units_l1"][1]))
        if n_layers >= 3:
            hidden_sizes.append(trial.suggest_int("n_units_l2", param_grid["n_units_l2"][0], param_grid["n_units_l2"][1]))

        self.activation   = nn.ReLU() if trial.suggest_categorical("activation", param_grid["activation"])=="relu" else nn.Tanh()
        self.dropout_rate = trial.suggest_categorical("dropout_rate", param_grid["dropout_rate"])

        layers = []
        in_dim = X.shape[1]
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(self.activation)
            layers.append(nn.Dropout(self.dropout_rate))
            in_dim = h
        self.net    = nn.Sequential(*layers)
        self.output = nn.Linear(in_dim, 1)

    def forward(self, x):
        x = self.net(x)
        return self.output(x)

# 4) Objective (with float/log LR, optimizer choice & weight_decay)
def objective(trial):
    model = OptunaRegNet(trial).to(device)

    lr  = trial.suggest_categorical("learning_rate", param_grid["learning_rate"])
    wd  = trial.suggest_categorical("weight_decay", param_grid["weight_decay"])
    bs  = trial.suggest_int("batch_size", param_grid["batch_size"][0], param_grid["batch_size"][1])

    # choose optimizer
    opt_name = trial.suggest_categorical("optimizer", param_grid["optimizer"])
    optimizer = (torch.optim.Adam if opt_name=="adam" else torch.optim.SGD)(
        model.parameters(), lr=lr, weight_decay=wd
    )

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False)
    loss_fn      = nn.MSELoss(reduction='sum')

    # training
    try:
        for _ in range(30):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                outputs = model(xb)
                loss = loss_fn(outputs, yb)
                
                # Check for NaN loss
                if torch.isnan(loss).any():
                    print(f"NaN loss detected during training with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN loss encountered")
                
                loss.backward()
                
                # Gradient clipping to prevent exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                # Check for NaN weights after update
                for param in model.parameters():
                    if torch.isnan(param).any():
                        print(f"NaN weights detected after update with params: {trial.params}")
                        raise optuna.exceptions.TrialPruned("NaN weights encountered")

        # evaluation
        model.eval()
        total_loss = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                
                # Check for NaN outputs
                if torch.isnan(outputs).any():
                    print(f"NaN outputs detected during evaluation with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN outputs encountered")
                
                batch_loss = loss_fn(outputs, yb).item()
                
                # Check for NaN loss
                if np.isnan(batch_loss):
                    print(f"NaN loss detected during evaluation with params: {trial.params}")
                    raise optuna.exceptions.TrialPruned("NaN evaluation loss encountered")
                
                total_loss += batch_loss

        avg_mse = total_loss / len(test_ds)
        
        # Final check for NaN result
        if np.isnan(avg_mse):
            print(f"NaN final MSE with params: {trial.params}")
            raise optuna.exceptions.TrialPruned("NaN final MSE encountered")
            
        trial.set_user_attr("mse", avg_mse)
        return -avg_mse  # maximize negative MSE
    
    except Exception as e:
        print(f"Error in trial with params {trial.params}: {str(e)}")
        raise optuna.exceptions.TrialPruned(f"Trial failed with error: {str(e)}")

# 5) Logging callback
results = []
def logging_callback(study, trial):
    entry = trial.params.copy()
    entry["neg_mse"]      = trial.value
    entry["trial_number"] = trial.number
    entry["trial_end"]    = time.time()
    entry["mse"]          = trial.user_attrs["mse"]
    results.append(entry)
    
    # Update progress bar based on elapsed time
    elapsed = time.time() - start_time
    pbar.n = min(elapsed, MAX_RUNTIME)
    pbar.refresh()


In [3]:
import optuna
from optuna.samplers import RandomSampler

# Maximum runtime per random search (1.75 hours in seconds)
MAX_RUNTIME = 1.75 * 60 * 60

# Seeds for reproducibility
random_seeds = [0, 0, 4]

start_time = time.time()
# Function to run a single random search with time limit
def run_random_search(seed_idx):
    global results
    
    # Set seed for reproducibility
    np.random.seed(random_seeds[seed_idx])
    torch.manual_seed(random_seeds[seed_idx])
    
    # Reset results for this run
    results = []
    
    # Create random sampler
    sampler = RandomSampler(seed=random_seeds[seed_idx])
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    # Number of trials for random search
    n_trials = 450
    
    print(f"Running random search {seed_idx+1}/3 with seed {random_seeds[seed_idx]}")
    print(f"Will run up to {n_trials} configurations")
    
    # Setup progress bar
    global pbar
    pbar = tqdm(total=MAX_RUNTIME, desc=f"Random Search (seed={random_seeds[seed_idx]})")
    
    # Record start time
    start_time = time.time()
    results.append({"start_time": start_time})
    
    # Run optimization with time limit
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=5,
        callbacks=[logging_callback],
        timeout=MAX_RUNTIME
    )
    
    # Close progress bar
    pbar.close()
    
    # Save results
    with open(f"results/RANDOM/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"Random search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)
    
    return study.best_value, study.best_trial.params

# Create directory if it doesn't exist
import os
os.makedirs("results/RANDOM", exist_ok=True)

# Run all three random searches sequentially
best_random_configs = []
for i in [2]:
    best_value, best_params = run_random_search(i)
    best_random_configs.append((best_value, best_params))


[I 2025-06-17 07:04:11,043] A new study created in memory with name: no-name-2aa055c7-92f2-4dca-a20c-86d7ecd6b378


Running random search 3/3 with seed 4
Will run up to 450 configurations


Random Search (seed=4):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 07:04:50,816] Trial 4 finished with value: -0.5224361770836882 and parameters: {'n_layers': 3, 'n_units_l0': 95, 'n_units_l1': 209, 'n_units_l2': 70, 'activation': 'tanh', 'dropout_rate': 0.2, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 215, 'optimizer': 'sgd'}. Best is trial 4 with value: -0.5224361770836882.
[I 2025-06-17 07:05:00,946] Trial 0 finished with value: -0.32722043390421907 and parameters: {'n_layers': 2, 'n_units_l0': 150, 'n_units_l1': 63, 'activation': 'relu', 'dropout_rate': 0.2, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 148, 'optimizer': 'adam'}. Best is trial 0 with value: -0.32722043390421907.
[I 2025-06-17 07:05:04,187] Trial 2 finished with value: -0.3453927187956581 and parameters: {'n_layers': 1, 'n_units_l0': 136, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 136, 'optimizer': 'adam'}. Best is trial 0 with value: -0.32722043390421907.
[I 2025-06-17 07:05:

Random search 3 completed
Best negative MSE: -0.2666402794131937
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 145, 'n_units_l1': 144, 'n_units_l2': 127, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 80, 'optimizer': 'adam'}
Runtime: 84.57 minutes
--------------------------------------------------


In [4]:
import optuna
from optuna.samplers import TPESampler
import time
import json
import numpy as np
import torch
from tqdm.notebook import tqdm

# Maximum runtime per TPE search (1.75 hours in seconds)
MAX_RUNTIME = 1.75 * 60 * 60

# Seeds for reproducibility
tpe_seeds = [7, 8, 9]

# Function to run a single TPE search with time limit
def run_tpe_search(seed_idx):
    global results
    
    # Set seed for reproducibility
    np.random.seed(tpe_seeds[seed_idx])
    torch.manual_seed(tpe_seeds[seed_idx])
    
    # Reset results for this run
    results = []
    
    # Create TPE sampler
    sampler = TPESampler(seed=tpe_seeds[seed_idx])
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    # Number of trials for TPE search
    n_trials = 450
    
    print(f"Running TPE search {seed_idx+1}/3 with seed {tpe_seeds[seed_idx]}")
    print(f"Will run up to {n_trials} configurations")
    
    # Setup progress bar
    global pbar
    pbar = tqdm(total=MAX_RUNTIME, desc=f"TPE Search (seed={tpe_seeds[seed_idx]})")
    
    # Record start time
    start_time = time.time()
    results.append({"start_time": start_time})
    
    # Run optimization with time limit
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=5,
        callbacks=[logging_callback],
        timeout=MAX_RUNTIME
    )
    
    # Close progress bar
    pbar.close()
    
    # Save results
    with open(f"results/TPE/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"TPE search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)
    
    return study.best_value, study.best_trial.params

# Create directory if it doesn't exist
os.makedirs("results/TPE", exist_ok=True)

# Run all three TPE searches sequentially
best_tpe_configs = []
for i in range(3):
    best_value, best_params = run_tpe_search(i)
    best_tpe_configs.append((best_value, best_params))


[I 2025-06-17 08:28:45,052] A new study created in memory with name: no-name-ec5f2c29-f0b8-4f38-8f06-2e96ce321eb6


Running TPE search 1/3 with seed 7
Will run up to 450 configurations


TPE Search (seed=7):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 08:29:17,636] Trial 0 finished with value: -4.053715728050054 and parameters: {'n_layers': 2, 'n_units_l0': 65, 'n_units_l1': 37, 'activation': 'tanh', 'dropout_rate': 0.2, 'learning_rate': 0.0001, 'weight_decay': 0.0001, 'batch_size': 213, 'optimizer': 'sgd'}. Best is trial 0 with value: -4.053715728050054.
[I 2025-06-17 08:29:25,539] Trial 2 finished with value: -0.32328324632127153 and parameters: {'n_layers': 3, 'n_units_l0': 97, 'n_units_l1': 196, 'n_units_l2': 177, 'activation': 'tanh', 'dropout_rate': 0.2, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 249, 'optimizer': 'adam'}. Best is trial 2 with value: -0.32328324632127153.
[I 2025-06-17 08:29:26,724] Trial 1 finished with value: -0.43429558175478794 and parameters: {'n_layers': 2, 'n_units_l0': 211, 'n_units_l1': 145, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.01, 'weight_decay': 0.01, 'batch_size': 145, 'optimizer': 'sgd'}. Best is trial 2 with value: -0.32328324632127153.
[I 

TPE search 1 completed
Best negative MSE: -0.25535547975883927
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 251, 'n_units_l1': 176, 'n_units_l2': 252, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 56, 'optimizer': 'adam'}
Runtime: 105.90 minutes
--------------------------------------------------
Running TPE search 2/3 with seed 8
Will run up to 450 configurations


...

[I 2025-06-17 10:15:11,537] Trial 2 finished with value: -0.5692744587743005 and parameters: {'n_layers': 3, 'n_units_l0': 243, 'n_units_l1': 170, 'n_units_l2': 71, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 245, 'optimizer': 'sgd'}. Best is trial 2 with value: -0.5692744587743005.
[I 2025-06-17 10:15:17,731] Trial 3 finished with value: -0.32192781128624615 and parameters: {'n_layers': 2, 'n_units_l0': 187, 'n_units_l1': 126, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 241, 'optimizer': 'adam'}. Best is trial 3 with value: -0.32192781128624615.
[I 2025-06-17 10:15:26,484] Trial 0 finished with value: -0.48442400067813635 and parameters: {'n_layers': 2, 'n_units_l0': 223, 'n_units_l1': 176, 'activation': 'tanh', 'dropout_rate': 0.4, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'batch_size': 147, 'optimizer': 'adam'}. Best is trial 3 with value: -0.32192781128624615.

TPE search 2 completed
Best negative MSE: -0.25679521505222763
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 230, 'n_units_l1': 221, 'n_units_l2': 236, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 39, 'optimizer': 'adam'}
Runtime: 106.03 minutes
--------------------------------------------------
Running TPE search 3/3 with seed 9
Will run up to 450 configurations


TPE Search (seed=9):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 12:01:13,796] Trial 4 finished with value: -0.5949319185212602 and parameters: {'n_layers': 1, 'n_units_l0': 96, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'batch_size': 209, 'optimizer': 'sgd'}. Best is trial 4 with value: -0.5949319185212602.
[I 2025-06-17 12:01:17,795] Trial 0 finished with value: -0.28950100536494294 and parameters: {'n_layers': 3, 'n_units_l0': 191, 'n_units_l1': 135, 'n_units_l2': 181, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 245, 'optimizer': 'adam'}. Best is trial 0 with value: -0.28950100536494294.
[I 2025-06-17 12:01:23,833] Trial 2 finished with value: -0.5428269735371419 and parameters: {'n_layers': 1, 'n_units_l0': 37, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 0.01, 'batch_size': 147, 'optimizer': 'sgd'}. Best is trial 0 with value: -0.28950100536494294.
[I 2025-06-17 12:01:32,500] Trial 1 fin

TPE search 3 completed
Best negative MSE: -0.2680283090056375
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 111, 'n_units_l1': 204, 'n_units_l2': 55, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 98, 'optimizer': 'adam'}
Runtime: 85.76 minutes
--------------------------------------------------


In [5]:
# CMA-ES Optimization
import optuna
from optuna.samplers import CmaEsSampler
import time
import json
import numpy as np
import torch
from tqdm.notebook import tqdm
import os

# Maximum runtime per CMA-ES search (1.75 hours in seconds)
MAX_RUNTIME = 1.75 * 60 * 60

# Seeds for reproducibility
cmaes_seeds = [10, 11, 12]

# Function to run a single CMA-ES search with time limit
def run_cmaes_search(seed_idx):
    global results
    
    # Set seed for reproducibility
    np.random.seed(cmaes_seeds[seed_idx])
    torch.manual_seed(cmaes_seeds[seed_idx])
    
    # Reset results for this run
    results = []
    
    # Create CMA-ES sampler
    sampler = CmaEsSampler(
        sigma0=0.3,  # Initial step size
        seed=cmaes_seeds[seed_idx]
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    # Number of trials for CMA-ES search
    n_trials = 625
    
    print(f"Running CMA-ES search {seed_idx+1}/3 with seed {cmaes_seeds[seed_idx]}")
    print(f"Will run up to {n_trials} configurations")
    
    # Setup progress bar
    global pbar
    pbar = tqdm(total=MAX_RUNTIME, desc=f"CMA-ES Search (seed={cmaes_seeds[seed_idx]})")
    
    # Record start time
    start_time = time.time()
    results.append({"start_time": start_time})
    
    # Run optimization with time limit
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=5,
        callbacks=[logging_callback],
        timeout=MAX_RUNTIME
    )
    
    # Close progress bar
    pbar.close()
    
    # Save results
    with open(f"results/CMAES/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"CMA-ES search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)
    
    return study.best_value, study.best_trial.params

# Create directory if it doesn't exist
os.makedirs("results/CMAES", exist_ok=True)

# Run all three CMA-ES searches sequentially
best_cmaes_configs = []
for i in range(3):
    best_value, best_params = run_cmaes_search(i)
    best_cmaes_configs.append((best_value, best_params))

[I 2025-06-17 13:26:26,465] A new study created in memory with name: no-name-30f7c3a7-d054-4eea-a196-c673b4480a5d


Running CMA-ES search 1/3 with seed 10
Will run up to 625 configurations


CMA-ES Search (seed=10):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 13:27:07,217] Trial 4 finished with value: -0.4407915667912295 and parameters: {'n_layers': 1, 'n_units_l0': 97, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.01, 'weight_decay': 0.01, 'batch_size': 142, 'optimizer': 'sgd'}. Best is trial 4 with value: -0.4407915667912295.
[W 2025-06-17 13:27:07,254] The parameter 'n_layers' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalDistribution`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `CmaEsSampler`, if this independent sampling is intended behavior.
[W 2025-06-17 13:27:07,256] The parameter 'activation' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalDistri

CMA-ES search 1 completed
Best negative MSE: -0.27131763493367866
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 167, 'n_units_l1': 65, 'n_units_l2': 87, 'activation': 'relu', 'dropout_rate': 0.0, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 173, 'optimizer': 'adam'}
Runtime: 90.78 minutes
--------------------------------------------------
Running CMA-ES search 2/3 with seed 11
Will run up to 625 configurations


...

[I 2025-06-17 14:57:42,924] Trial 3 finished with value: -0.5165376547695131 and parameters: {'n_layers': 1, 'n_units_l0': 123, 'activation': 'tanh', 'dropout_rate': 0.4, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 214, 'optimizer': 'sgd'}. Best is trial 3 with value: -0.5165376547695131.
[W 2025-06-17 14:57:42,930] The parameter 'n_layers' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalDistribution`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `CmaEsSampler`, if this independent sampling is intended behavior.
[W 2025-06-17 14:57:42,931] The parameter 'n_units_l1' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalDis

CMA-ES search 2 completed
Best negative MSE: -0.2698109240032906
Best hyperparameters: {'n_layers': 3, 'n_units_l0': 150, 'n_units_l1': 176, 'n_units_l2': 56, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 122, 'optimizer': 'adam'}
Runtime: 96.43 minutes
--------------------------------------------------
Running CMA-ES search 3/3 with seed 12
Will run up to 625 configurations


CMA-ES Search (seed=12):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 16:34:11,922] Trial 3 finished with value: -0.48091347642647203 and parameters: {'n_layers': 1, 'n_units_l0': 163, 'activation': 'tanh', 'dropout_rate': 0.4, 'learning_rate': 0.001, 'weight_decay': 0.01, 'batch_size': 200, 'optimizer': 'adam'}. Best is trial 3 with value: -0.48091347642647203.
[W 2025-06-17 16:34:11,928] The parameter 'n_layers' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalDistribution`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `CmaEsSampler`, if this independent sampling is intended behavior.
[W 2025-06-17 16:34:11,929] The parameter 'n_units_l1' in trial#5 is sampled independently by using `RandomSampler` instead of `CmaEsSampler` (optimization performance may be degraded). `CmaEsSampler` does not support dynamic search space or `CategoricalD

CMA-ES search 3 completed
Best negative MSE: -0.26900015089863033
Best hyperparameters: {'n_layers': 2, 'n_units_l0': 162, 'n_units_l1': 96, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 156, 'optimizer': 'adam'}
Runtime: 99.47 minutes
--------------------------------------------------


In [ ]:
# GP Optimization
from optuna.samplers import GPSampler

# Maximum runtime per GP search (1.75 hours in seconds)
MAX_RUNTIME = 1.75 * 60 * 60

# Seeds for reproducibility
gp_seeds = [13, 14, 15]

# Function to run a single GP search with time limit
def run_gp_search(seed_idx):
    global results
    
    # Set seed for reproducibility
    np.random.seed(gp_seeds[seed_idx])
    torch.manual_seed(gp_seeds[seed_idx])
    
    # Reset results for this run
    results = []
    
    # Create GP sampler
    sampler = GPSampler(
        seed=gp_seeds[seed_idx],
        deterministic_objective=True
    )
    study = optuna.create_study(direction="maximize", sampler=sampler)
    
    # Number of trials for GP search
    n_trials = 425
    
    print(f"Running GP search {seed_idx+1}/3 with seed {gp_seeds[seed_idx]}")
    print(f"Will run up to {n_trials} configurations")
    
    # Setup progress bar
    global pbar
    pbar = tqdm(total=MAX_RUNTIME, desc=f"GP Search (seed={gp_seeds[seed_idx]})")
    
    # Record start time
    start_time = time.time()
    results.append({"start_time": start_time})
    
    # Run optimization with time limit
    study.optimize(
        objective,
        n_trials=n_trials,
        n_jobs=5,
        callbacks=[logging_callback],
        timeout=MAX_RUNTIME
    )
    
    # Close progress bar
    pbar.close()
    
    # Save results
    with open(f"results/GP/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"GP search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)
    
    return study.best_value, study.best_trial.params

# Create directory if it doesn't exist
os.makedirs("results/GP", exist_ok=True)

# Run all three GP searches sequentially
best_gp_configs = []
for i in range(3):
    best_value, best_params = run_gp_search(i)
    best_gp_configs.append((best_value, best_params))


C:\Users\16073\AppData\Local\Temp\ipykernel_43736\2018794469.py:22: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler = GPSampler(
[I 2025-06-17 18:13:07,920] A new study created in memory with name: no-name-2fd4d091-e998-4e10-95bd-03ba0175ca8a


Running GP search 1/3 with seed 13
Will run up to 425 configurations


GP Search (seed=13):   0%|          | 0/6300.0 [00:00<?, ?it/s]

[I 2025-06-17 18:13:41,595] Trial 1 finished with value: -0.47753143865008685 and parameters: {'n_layers': 3, 'n_units_l0': 135, 'n_units_l1': 46, 'n_units_l2': 144, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.01, 'weight_decay': 1e-06, 'batch_size': 202, 'optimizer': 'sgd'}. Best is trial 1 with value: -0.47753143865008685.
[I 2025-06-17 18:13:47,659] Trial 3 finished with value: -0.4462988944940789 and parameters: {'n_layers': 1, 'n_units_l0': 138, 'activation': 'relu', 'dropout_rate': 0.4, 'learning_rate': 0.01, 'weight_decay': 0.0001, 'batch_size': 147, 'optimizer': 'sgd'}. Best is trial 3 with value: -0.4462988944940789.
[I 2025-06-17 18:14:19,006] Trial 4 finished with value: -0.2907507934773615 and parameters: {'n_layers': 3, 'n_units_l0': 98, 'n_units_l1': 82, 'n_units_l2': 155, 'activation': 'tanh', 'dropout_rate': 0.0, 'learning_rate': 0.001, 'weight_decay': 1e-06, 'batch_size': 94, 'optimizer': 'adam'}. Best is trial 4 with value: -0.2907507934773615.
[I 20

In [ ]:
import sys
import os

seeds = [16, 17, 18]
# Add the hyperband_sampler directory to Python path
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Go up to HPOptimization root directory  
root_dir = os.path.dirname(os.path.dirname(os.path.dirname(current_dir)))
print(f"Root directory: {root_dir}")

# Add hyperband_sampler to path
hyperband_path = os.path.join(root_dir, "hyperband_sampler")
print(f"Adding to path: {hyperband_path}")

if hyperband_path not in sys.path:
    sys.path.append(hyperband_path)

# Now import the class
from hyperband_study import HyperbandStudy

# Create directory if it doesn't exist
os.makedirs("results/HYPERBAND", exist_ok=True)

# Define reduction factor
r = 3
os.makedirs(f"results/HYPERBAND/n={r}", exist_ok=True)

for seed_idx, seed in enumerate(seeds):
    # 4) Objective (modified for Hyperband)
    def objective(trial):
        model = OptunaHousingNet(trial).to(device)
        lr = trial.suggest_categorical("learning_rate", param_grid["learning_rate"])
        bs = trial.suggest_categorical("batch_size", param_grid["batch_size"])
        
        # Resource parameter controlled by Hyperband
        epochs = trial.suggest_int("resource", 1, 100)
        
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
        test_loader  = DataLoader(test_ds, batch_size=bs, shuffle=False)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.MSELoss(reduction='sum')
        
        # Train for the number of epochs determined by Hyperband
        for _ in range(int(epochs)):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
        
        # Evaluate
        model.eval()
        total_loss = 0
        total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                loss = loss_fn(outputs, yb).item()
                total_loss += loss
                total += yb.size(0)
        mse = total_loss / total
        trial.set_user_attr('mse', mse)
        return -mse  # We want to maximize negative MSE (minimize MSE)

    # Clear previous results
    results = []
    
    # 6) Create Hyperband study
    hyperband_study = HyperbandStudy(
        directions="maximize",
        min_resource=3, 
        max_resource=30, 
        reduction_factor=r,
        hyperband_iterations=55,
        sampler_seed=seed
    )

    # Show bracket structure
    print(f"Running Hyperband search {seed_idx+1}/3 with seed {seed}")
    print("Hyperband bracket structure:")
    print(f"Total trials needed: {hyperband_study.n_trials}")
    for j, bracket in enumerate(hyperband_study.sampler._brackets):
        print(f"Bracket {j}: {len(bracket['rungs'])} rungs")
        for k, rung in enumerate(bracket['rungs']):
            print(f"  Rung {k}: {rung['n_configs']} trials × {rung['resource']} epochs")

    # 7) Prepare logging and run optimization
    pbar = tqdm(total=hyperband_study.n_trials, desc=f"Hyperband Search (seed={seed})")
    
    # Record start time
    start_time = time.time()
    results.append({"start_time": start_time})

    # Add callback and run
    study = hyperband_study.optimize(
        objective=objective, 
        callbacks=[logging_callback], 
        n_jobs=5,
        timeout=MAX_RUNTIME
    )
    pbar.close()

    # 8) Save results to JSON
    with open(f"results/HYPERBAND/n={r}/results_{seed_idx+1}.json", "w") as f:
        json.dump(results, f, indent=2)
    
    # Print best result
    print(f"Hyperband search {seed_idx+1} completed")
    print(f"Best negative MSE: {study.best_value}")
    print(f"Best hyperparameters: {study.best_trial.params}")
    print(f"Runtime: {(time.time() - start_time) / 60:.2f} minutes")
    print("-" * 50)